# Foodpanda Delivery Delay Prediction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score
)

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("E:\\Deep Learning\\Foodpanda\\dataset\\Foodpanda.csv")

date_cols = ["signup_date", "order_date", "last_order_date", "rating_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

text_cols = ["customer_id", "gender", "age", "city", "order_id",
             "restaurant_name", "dish_name", "category",
             "payment_method", "churned", "delivery_status"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

df["rating_status"] = np.where(df["rating"].isna(), "Not Rated", "Rated")
df["order_value"] = df["quantity"] * df["price"]
df = df.rename(columns={"age": "age_group"})

reference_date = df["last_order_date"].max()
df["tenure_days"] = (reference_date - df["signup_date"]).dt.days
df["recency_days"] = (reference_date - df["last_order_date"]).dt.days
df["order_month"] = df["order_date"].dt.month
df["order_year"] = df["order_date"].dt.year
df["order_day"] = df["order_date"].dt.day
df["order_weekday"] = df["order_date"].dt.day_name()

print("Shape:", df.shape)
df.head()

In [ ]:
delay_df = df.copy()
delay_df["is_delayed"] = np.where(delay_df["delivery_status"].str.lower() == "delayed", 1, 0)

features_delay = [
    "gender", "age_group", "city", "restaurant_name", "dish_name",
    "category", "quantity", "price", "payment_method",
    "order_frequency", "loyalty_points", "rating",
    "tenure_days", "recency_days", "order_month", "order_day"
]

delay_data = delay_df[features_delay + ["is_delayed"]].copy()
delay_data["rating"] = delay_data["rating"].fillna(delay_data["rating"].median())
delay_data.head()

In [ ]:
delay_encoders = {}
for col in delay_data.select_dtypes(include="object").columns:
    le = LabelEncoder()
    delay_data[col] = le.fit_transform(delay_data[col])
    delay_encoders[col] = le

X_delay = delay_data.drop(columns=["is_delayed"])
y_delay = delay_data["is_delayed"]

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_delay, y_delay, test_size=0.2, random_state=42, stratify=y_delay
)

In [ ]:
delay_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)
delay_model.fit(X_train_d, y_train_d)
y_pred_delay = delay_model.predict(X_test_d)

print("Accuracy:", accuracy_score(y_test_d, y_pred_delay))
print(classification_report(y_test_d, y_pred_delay))

In [ ]:
delay_importance = pd.Series(delay_model.feature_importances_, index=X_delay.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
delay_importance.head(10).plot(kind="bar")
plt.title("Top 10 Feature Importances - Delay Prediction")
plt.ylabel("Importance")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()